# 01 Baseline: EDA, sentence splitting, aspect tagging, TF-IDF baseline

Dataset: TripAdvisor hotel reviews (Kaggle). Save the CSV as `data/raw/tripadvisor_hotel_reviews.csv` first.
Run from the `notebooks/` folder.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

from src.pipeline import (load_reviews, clean_text, rating_to_sentiment, split_sentences,
                          add_difficulty_flags, sample_for_labelling, ASPECT_KEYWORDS)

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "raw" / "tripadvisor_hotel_reviews.csv"
OUT = ROOT / "outputs"
SEED = 42

## 1. Load and inspect

In [ ]:
df = load_reviews(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df["n_words"] = df["review"].str.split().map(len)
print(df["n_words"].describe())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df["rating"].value_counts().sort_index().plot.bar(ax=ax[0], title="Rating distribution")
df["n_words"].clip(upper=800).plot.hist(bins=40, ax=ax[1], title="Review length (words, clipped at 800)")
plt.tight_layout()
plt.savefig(OUT / "eda_ratings_lengths.png", dpi=150)
plt.show()

## 2. Clean and create review-level labels
Assumption: 4-5 stars = positive, 3 = neutral, 1-2 = negative.

In [ ]:
df["review"] = df["review"].map(clean_text)
df["sentiment"] = df["rating"].map(rating_to_sentiment)
df["sentiment"].value_counts()

## 3. Sentence splitting, aspect tagging and difficulty flags

In [ ]:
sents = split_sentences(df)
flagged = add_difficulty_flags(sents)
print(len(sents), "sentences from", len(df), "reviews")
print("Sentences with at least one aspect:", (flagged["n_aspects"] >= 1).mean().round(3))
print("Negation:", flagged["has_negation"].mean().round(3),
      "| Contrast:", flagged["has_contrast"].mean().round(3),
      "| Multi-aspect:", flagged["is_multi_aspect"].mean().round(3))

aspect_counts = flagged.explode("aspects")["aspects"].value_counts()
aspect_counts.plot.barh(title="Sentences per aspect (keyword lexicon)")
plt.tight_layout(); plt.savefig(OUT / "aspect_counts.png", dpi=150); plt.show()

## 4. Review-level baseline: TF-IDF + logistic regression
This is the 'standard sentiment baseline'. It predicts one label per review, so it cannot separate aspects.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["sentiment"], test_size=0.2, random_state=SEED, stratify=df["sentiment"])

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(vec.fit_transform(X_train), y_train)
pred = clf.predict(vec.transform(X_test))

print(classification_report(y_test, pred, digits=3))
macro_f1 = f1_score(y_test, pred, average="macro")
pd.DataFrame({"model": ["tfidf_logreg_review_level"], "macro_f1": [macro_f1]}).to_csv(OUT / "baseline_metrics.csv", index=False)
macro_f1

## 5. Export a sheet for hand-labelling (about 200 sentences)
Sampling is roughly balanced across negation, contrast, multi-aspect and plain sentences.
Fill in `gold_aspect`, `gold_sentiment` (positive / negative / neutral toward that aspect) and `difficulty` by hand.

In [ ]:
sheet = sample_for_labelling(flagged, n=200, seed=SEED)
sheet.to_csv(ROOT / "data" / "labelled" / "labelling_sheet.csv", index=False)
print(sheet["sample_group"].value_counts())
sheet.head(10)

## Next steps
1. Open `data/labelled/labelling_sheet.csv`, label the sentences, save as `labelled_sentences.csv`.
2. Add the transformer model (Hugging Face sentiment pipeline) and compare per difficulty category.
3. Build the demo (notebook cell or Gradio app).